# Pool Classification
The goal of the challenge is to train a pool classifier that can determine whether images of plots contain pools.

### Import the neccessary libraries

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import glob
import datetime

from model import PoolClassifier
from dataloader import SwimmingPoolDataset

### Define arguments

In [2]:
#DATA_DIR
BATCH_SIZE = 64
NUM_EPOCHS = 100
POOL_THRESH = 0.5
WEIGHTS = ""
DEVICE = "cuda"
NUM_WORKERS = 64
LR = 1e-4

### Initialize model, dataloader and optimizer

In [ ]:
training_set = SwimmingPoolDataset(DATA_DIR, val=False)
trainloader = torch.utils.data.DataLoader(training_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

validation_set = SwimmingPoolDataset(DATA_DIR, val=True)
valloader = torch.utils.data.DataLoader(validation_set, batch_size=int(BATCH_SIZE / 9), shuffle=True, num_workers=NUM_WORKERS)

classifier = PoolClassifier()
if WEIGHTS: classifier.load_state_dict(torch.load(WEIGHTS, weights_only=True))
classifier.to(DEVICE)

optimizer = torch.optim.Adam(params=classifier.parameters(), lr=LR)

### Train classifier

In [ ]:
train_iter, val_iter = 0, 0
for epoch in range(NUM_EPOCHS):
    for item in trainloader:
        optimizer.zero_grad()

        train_iter += 1
        image_batch, target_batch = item
        image_batch = image_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        result = classifier(image_batch) # bs
        loss = F.binary_cross_entropy_with_logits(result, target_batch, reduction="sum")

        loss.backward()

        with torch.no_grad():
            pool_infers = torch.where(result.sigmoid() > POOL_THRESH, 1., 0.)
            num_pools = target_batch.sum()
            if num_pools > 0:
                det_pools = (pool_infers * target_batch).sum() / num_pools
                miss_pools = ((1-pool_infers) * target_batch).sum() / num_pools
                false_pools = (pool_infers * (1-target_batch)).sum() / pool_infers.sum()
            else:
                det_pools, miss_pools, false_pools = 0, 0, 0

        optimizer.step()

        if train_iter % 100 == 1:
            print(f"Epoch: {epoch}, Train Iter: {train_iter}, Loss: {loss:.2f}, Det Pools: {det_pools:.2f}, Miss Pools: {miss_pools:.2f}, False Pos: {false_pools:.2f}")

    if epoch % 5 == 0:
        for item in valloader:
            with torch.no_grad():
                val_iter += 1
                image_batch, target_batch = item
                image_batch = image_batch.to(DEVICE) # bs,9,c,h,w
                target_batch = target_batch.to(DEVICE) # bs

                image_batch = image_batch.reshape(-1, 3, 417, 417)
                result = classifier(image_batch).reshape(-1, 9) # bs,9
                pool_infers = torch.where(result.sigmoid() > FLAGS.pool_thresh, 1., 0.)
                pool_infers = pool_infers.max(dim=1)[0] # bs

                num_pools = target_batch.sum()
                if num_pools > 0:
                    det_pools = (pool_infers * target_batch).sum() / num_pools
                    miss_pools = ((1-pool_infers) * target_batch).sum() / num_pools
                    false_pools = (pool_infers * (1-target_batch)).sum() / pool_infers.sum()
                else:
                    det_pools, miss_pools, false_pools = 0, 0, 0

                if val_iter % 100 == 1:
                    print(f"Val Iter: {val_iter}, Loss: {loss:.2f}, Det Pools: {det_pools:.2f}, Miss Pools: {miss_pools:.2f}, False Pos: {false_pools:.2f}")

        torch.save(classifier.state_dict(),f"weights/{FLAGS.exp}_{epoch}.pt")